In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from pathlib import Path
from tqdm import tqdm
import numpy as np

In [2]:
# CONSTANTS:
ALPHAS = [0.01, 0.025, 0.05, 0.1, 0.25, 0.5]

# I should probably export the max_ind_set indices into a file so the selection is fixed and reproducible
# We should also fix our phenotype manifest so our arbitrary index is fixed

In [3]:
phenotype_manifest = pd.read_csv("phenotype_manifest.csv", usecols=[
    'description',
    'in_max_independent_set',
    'phenocode',
    'filename'
    ])

In [4]:
phenotype_manifest['description'] = phenotype_manifest['description'].fillna(phenotype_manifest['phenocode'])
max_ind_set = phenotype_manifest[phenotype_manifest['in_max_independent_set']]

In [5]:
exclude_phenotypes = []

In [6]:
with open("exclude_phenotypes.txt", "r") as file:
    for line in file:
        if line[0]=='#':
            continue
        exclude_phenotypes.append(int(line.strip().split()[0]))

In [7]:
expected_folders = []

for index, row in max_ind_set.iterrows():
    if index in exclude_phenotypes:
        expected_folders.append(np.nan)
    else:
        expected_folders.append(os.path.splitext(os.path.basename(row['filename']))[0])

In [8]:
metadata_columns = [
    "index",
    "description",
    "alpha",
    "dir_name",
    "num_snps_found",
    "num_coding_snps_found",
    "num_overlapping_snps",
    "num_overlapping_loci",
    "num_original_list",
    "num_original_coding_snps",
    "p_value_threshold",
    "percentage_loci_recovered",
    "ld_based_clumping",
    "r2_threshold",
    "kb_radius",
    "window_size"]
collected_metadata_df = pd.DataFrame(columns=metadata_columns)
collected_metadata_df['description'] = max_ind_set['description']
collected_metadata_df['index'] = max_ind_set.index
collected_metadata_df['dir_name'] = expected_folders
collected_metadata_df = collected_metadata_df.loc[collected_metadata_df.index.repeat(len(ALPHAS))].reset_index(drop=True)
collected_metadata_df['alpha'] = ALPHAS * len(max_ind_set)

In [9]:
missing_indeces = []

for idx, row in collected_metadata_df.iterrows():
    alpha_folder = f"deepcast_alpha_ablation/results_{str(row['alpha'])[0]+str(row['alpha'])[2:]}"
    folder_path = f"{alpha_folder}/{row['dir_name']}"
    if not Path(folder_path):
        raise ValueError("No phecode directories found in 'patho_phenotypes/deepcast_phenotypes' folder")
    metadata_path = f"{folder_path}/metadata_df.csv"
    try:
        # Read the small single-row CSV
        temp_df = pd.read_csv(metadata_path)
        
        # Now update the corresponding fields in df1
        for col in temp_df.columns:
            if col in collected_metadata_df.columns:
                collected_metadata_df.at[idx, col] = temp_df.iloc[0][col]
                
    except FileNotFoundError:
        print(f"File {metadata_path} not found, skipping.")
        missing_indeces.append(idx)    

File deepcast_alpha_ablation/results_0025/biomarkers-30610-both_sexes-irnt.tsv/metadata_df.csv not found, skipping.
File deepcast_alpha_ablation/results_0025/biomarkers-30620-both_sexes-irnt.tsv/metadata_df.csv not found, skipping.
File deepcast_alpha_ablation/results_0025/biomarkers-30630-both_sexes-irnt.tsv/metadata_df.csv not found, skipping.
File deepcast_alpha_ablation/results_0025/biomarkers-30640-both_sexes-irnt.tsv/metadata_df.csv not found, skipping.
File deepcast_alpha_ablation/results_0025/biomarkers-30690-both_sexes-irnt.tsv/metadata_df.csv not found, skipping.
File deepcast_alpha_ablation/results_0025/biomarkers-30700-both_sexes-irnt.tsv/metadata_df.csv not found, skipping.
File deepcast_alpha_ablation/results_0025/biomarkers-30710-both_sexes-irnt.tsv/metadata_df.csv not found, skipping.
File deepcast_alpha_ablation/results_0025/biomarkers-30720-both_sexes-irnt.tsv/metadata_df.csv not found, skipping.
File deepcast_alpha_ablation/results_0025/biomarkers-30730-both_sexes-ir

In [10]:
collected_metadata_df['num_snps_found'].isna().sum()

np.int64(109)

In [11]:
collected_metadata_df['dir_name'].isna().sum()

np.int64(84)

In [12]:
collected_metadata_df.to_csv("collected_metadata.csv")

In [ ]:
phenotype_dirs = [d for d in Path("patho_phenotypes/deepcast_phenotypes").iterdir() if d.is_dir() and d.name == "outputs"]
phenotype_dirs = [d for d in Path(f"{alpha_folder}/deepcast_phenotypes").iterdir() if d.is_dir() and d.name == "outputs"]

In [ ]:
def load_metadata():
    """Load metadata from all phenotype directories."""
    metadata_list = []
    phenotype_dirs = [d for d in Path("patho_phenotypes/deepcast_phenotypes").iterdir() if d.is_dir() and d.name.startswith("phecode-")]
    
    if not phenotype_dirs:
        raise ValueError("No phecode directories found in 'patho_phenotypes/deepcast_phenotypes' folder")
    
    print(f"Found {len(phenotype_dirs)} phecode directories")
    
    for pheno_dir in tqdm(phenotype_dirs, desc="Loading metadata"):
        metadata_file = pheno_dir / "metadata_df.csv"
        if metadata_file.exists():
            try:
                df = pd.read_csv(metadata_file)
                df['phenotype'] = pheno_dir.name
                metadata_list.append(df)
            except Exception as e:
                print(f"Error loading {metadata_file}: {e}")
        else:
            print(f"No metadata file found in {pheno_dir}")
    
    if not metadata_list:
        raise ValueError("No valid metadata files found in any phecode directory")
    
    print(f"Successfully loaded metadata from {len(metadata_list)} directories")
    return pd.concat(metadata_list, ignore_index=True)

In [ ]:
# Set style
plt.style.use('seaborn-v0_8')  # Updated style name
sns.set_theme(style="whitegrid")
sns.set_palette("Set2")